# パターン認識特論 第2回 演習：ニューラルネットワークの基礎アーキテクチャ

講義テキストの式を，そのまま NumPy で動かして確かめる．

| 節 | テーマ | スライド |
|---|---|---|
| §1 | ユニットと層の計算 | p.4–6 |
| §2 | 活性化関数と微分 | p.8–9 | 
| §3 | MLP の順伝播と表現変換 | p.11–13 | 
| §4 | 損失，ソフトマックスと交差エントロピー | p.14–16 |
| §5 | 誤差信号 $\boldsymbol{p}-\boldsymbol{y}$ の検証 | p.17 |
| §6 | 埋め込み | p.18 |
| §7 | （任意）PyTorch で同じことをする | テキスト付録B |

**使い方**
- セルを選んで **Shift + Enter** で実行する．**必ず上から順に**実行する（下のセルは上のセルで作った変数や関数を使う）．
- 結果がおかしくなったら，メニューの「Kernel → Restart Kernel and Run All Cells」で最初からやり直す．
- **やってみよう** は，先に予想してから，`# TODO` の行を書き換えて実行する．`# TODO` がないものは，実行して結果を読む．

環境：Python 3 ＋ NumPy ＋ Matplotlib．§7 のみ PyTorch（入っていなければ読み飛ばす）．

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)   # 数値を小数 4 桁で表示する
rng = np.random.default_rng(0)                    # 乱数（毎回同じ結果になるよう種を固定）

## §1 ユニットと層の計算（p.4–5）

$$u = \boldsymbol{w}^\top \boldsymbol{x} + b, \qquad z = h(u)$$

スライドの数値例：$\boldsymbol{x}=(2,1)^\top$，$\boldsymbol{w}=(0.5,-1)^\top$，$b=0.2$，$h=\mathrm{ReLU}$．
$\boldsymbol{w}^\top\boldsymbol{x}$ は内積 $w_1x_1+w_2x_2$．NumPy では `w @ x` と書く．

In [ ]:
def relu(u):
    return np.maximum(0.0, u)      # 正ならそのまま，負なら 0

x = np.array([2.0, 1.0])
w = np.array([0.5, -1.0])
b = 0.2
u = w @ x + b          # 重み付き和（内積）＋バイアス： 0.5×2 + (−1)×1 + 0.2
z = relu(u)            # 活性化関数
print(f"u = {u:.3f}, z = ReLU(u) = {z:.3f}")   # スライドと同じ 0.2, 0.2 になるはず

**やってみよう 1-1**　入力の 2 番目の成分 $x_2$ を 1 から 2 に変えると $u$ と $z$ はどうなるか．先に予想してから実行する
（$w_2=-1$ なので，$x_2$ を増やすと総入力は**下がる**）．

In [ ]:
x_new = np.array([2.0, 1.0])   # TODO: 2 番目の成分を 2.0 に変える（Python の添字は 0 から数えるので x_new[1] が x_2）
u_new = w @ x_new + b
print(f"u = {u_new:.3f}, z = {relu(u_new):.3f}")

In [ ]:
# 解答
x_new = np.array([2.0, 2.0])
u_new = w @ x_new + b
print(f"u = {u_new:.3f}, z = {relu(u_new):.3f}")   # u = -0.8 → ReLU で z = 0（このユニットの出力は 0 になる）

### ユニットを並べて層にする：重み行列 $W$（p.6）

同じ入力を受け取るユニットを 3 個並べる．ユニット $i$ の重みベクトルを第 $i$ 行に積んだものが **重み行列** $W$（ユニット数 × 入力数）．

$$\boldsymbol{u} = W\boldsymbol{x} + \boldsymbol{b},\qquad \boldsymbol{z} = h(\boldsymbol{u})$$

スライドの数値例：$W$ の行が $(0.5,-1)$，$(1,1)$，$(-1,2)$，$\boldsymbol{b}=(0.2, 0, -0.5)^\top$．第 1 行は上のユニットと同じ．

In [ ]:
x = np.array([2.0, 1.0])
W = np.array([[0.5, -1.0],     # ユニット 1 の重みベクトル（上の w と同じ）
              [1.0,  1.0],     # ユニット 2
              [-1.0, 2.0]])    # ユニット 3
b_vec = np.array([0.2, 0.0, -0.5])
print("W の形 =", W.shape, "（ユニット数 3 × 入力数 2）")

# (1) ユニットを 1 個ずつ計算する（W[i] は W の第 i 行）
for i in range(3):
    print(f"ユニット {i + 1}: u = w_{i + 1}・x + b_{i + 1} = {W[i] @ x + b_vec[i]:+.1f}")

# (2) 行列とベクトルの積 1 回でまとめて計算する（結果は同じ）
u_vec = W @ x + b_vec
z_vec = relu(u_vec)                # 活性化関数は成分ごとに適用する
print("u = W x + b =", u_vec, " z = ReLU(u) =", z_vec)   # u = (0.2, 3, -0.5)，z = (0.2, 3, 0)

**やってみよう 1-2（p.5 のクイズ）**　入力 4 個，ユニット 3 個の層を作る．`W` の形はどうなるか．乱数で作って確かめる．
次に，形を `(4, 3)` に変えて実行する．**赤いエラーが出れば正解**（行列とベクトルの形が合わない）．確かめたら `(3, 4)` に戻す．

In [ ]:
W4 = rng.normal(size=(3, 4))       # TODO: 形を (4, 3) に変えるとどうなるか（確かめたら戻す）
x4 = np.array([1.0, 0.0, -1.0, 2.0])
print("W の形 =", W4.shape, " u の形 =", (W4 @ x4).shape)   # 行＝ユニット，列＝入力．u はユニットの数だけ出る

## §2 活性化関数と微分（p.7–10）

シグモイド・tanh・ReLU・GELU と，その微分を式どおりに実装して描く（微分は GELU を除く 3 つ）．
下のセルの前半は関数の定義，後半は描画（左：関数，右：微分）．描画の書き方は読み飛ばしてよい．

In [ ]:
from math import erf, sqrt

def sigmoid(u):
    return 1.0 / (1.0 + np.exp(-u))

def d_sigmoid(u):                  # シグモイドの微分 σ(1−σ)
    s = sigmoid(u)
    return s * (1.0 - s)

def d_tanh(u):                     # tanh の微分 1 − tanh²
    return 1.0 - np.tanh(u) ** 2

def d_relu(u):                     # ReLU の微分：u > 0 なら 1.0，それ以外は 0.0
    return (u > 0).astype(float)

# GELU に使う Φ(u)：0 から 1 へ滑らかに増える関数．この 1 行は読み飛ばしてよい
Phi = np.vectorize(lambda t: 0.5 * (1.0 + erf(t / sqrt(2.0))))

def gelu(u):
    return u * Phi(u)

us = np.linspace(-4, 4, 401)                       # −4 から 4 までの 401 点
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))    # 左右 2 枚の図．ax[0] が左，ax[1] が右
ax[0].plot(us, sigmoid(us), ':', label='sigmoid')  # ':' 点線，'-.' 一点鎖線，'-' 実線，'--' 破線
ax[0].plot(us, np.tanh(us), '-.', label='tanh')
ax[0].plot(us, relu(us), '-', label='ReLU')
ax[0].plot(us, gelu(us), '--', label='GELU')
ax[0].set_title('h(u)'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(us, d_sigmoid(us), ':', label="sigmoid'")
ax[1].plot(us, d_tanh(us), '-.', label="tanh'")
ax[1].plot(us, d_relu(us), '-', label="ReLU'")
ax[1].set_title("h'(u)"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("sigmoid'(u) の最大値 =", d_sigmoid(0.0), "（u=0 で 0.25）")
print("0.25 を 10 層分掛けると =", 0.25 ** 10)

**やってみよう 2-1（p.9 のクイズの続き）**　シグモイドの微分の最大値 0.25 を 20 層分，50 層分掛けるといくらになるか．
ReLU の正の側（微分 1）なら何層でも 1 のままであることも確かめる．

In [ ]:
for n_layers in (10, 20, 50):       # TODO: 100 を足してみる
    print(f"{n_layers:3d} 層: sigmoid 側 {0.25 ** n_layers:.2e}, ReLU 側 {1.0 ** n_layers:.1f}")

**やってみよう 2-2（発展）：dying ReLU（p.10 の補足）**

ReLU の層で，「どの入力データに対しても出力が 0 のユニット」（更新されなくなったユニット）を数える．
これは学習の再現ではなく，**バイアスが負に大きくなった状態を人工的に作った模擬**である．
`shift` が大きいほどバイアスが負に大きい．`shift=0` では 0 個，`shift=4` ではほとんどのユニットが該当する．

In [ ]:
def count_dead_units(shift, n_hidden=256, n_samples=1000, d_in=10):
    X = rng.normal(size=(n_samples, d_in))                 # 入力データ 1000 件（1 行が 1 件）
    W = rng.normal(size=(n_hidden, d_in)) / np.sqrt(d_in)  # 256 ユニットの重み行列
    U0 = X @ W.T                                           # 各データ・各ユニットの重み付き和（1000×256）
    b = -shift * U0.std(axis=0)                            # バイアスを負にずらす（ユニットごとのばらつきの shift 倍）
    Z = relu(U0 + b)
    dead = np.all(Z == 0, axis=0)                          # axis=0：データ方向に調べる ＝ ユニットごとに「全データで 0 か」
    return int(dead.sum())

for shift in (0.0, 2.0, 3.0, 4.0):     # TODO: 値を変えて試す
    print(f"shift={shift}: 出力が常に 0 のユニット {count_dead_units(shift)} / 256")

## §3 MLP の順伝播と表現変換（p.11–13）

$$\boldsymbol{z}^{(l)} = h\left(W^{(l)}\boldsymbol{z}^{(l-1)} + \boldsymbol{b}^{(l)}\right),\qquad \boldsymbol{z}^{(0)}=\boldsymbol{x}$$

p.11 の図と同じ 2–3–2 の MLP（各層のユニット数が 2，3，2）を，小さな数値で計算する．
手計算の結果：$\boldsymbol{u}^{(1)}=(-1,\ 1.6,\ 2.5)$，$\boldsymbol{z}^{(1)}=(0,\ 1.6,\ 2.5)$，$\boldsymbol{u}^{(2)}=(-2.5,\ 5.7)$．

In [ ]:
W1 = np.array([[1.0, -1.0],
               [0.5,  0.5],
               [-1.0, 2.0]])          # 3×2：第1層（入力2 → 隠れ3）
b1 = np.array([0.0, 0.1, -0.5])
W2 = np.array([[1.0, 0.0, -1.0],
               [0.0, 2.0,  1.0]])     # 2×3：第2層（隠れ3 → 出力2）
b2 = np.array([0.0, 0.0])

x = np.array([1.0, 2.0])
u1 = W1 @ x + b1
z1 = relu(u1)
u2 = W2 @ z1 + b2                     # 出力層は活性化関数なし（そのまま出す）とする
print("u1 =", u1, " z1 =", z1)
print("u2 =", u2)

n_params = W1.size + b1.size + W2.size + b2.size
print("パラメータ数 =", n_params, "（重み 6+6=12，バイアス 3+2=5）")   # p.10 のクイズ：線は 12 本

### 表現変換を見る（p.13）

MLP ＝「隠れ層による変換」＋「出力層の線形識別器」（p.13）を，実際に訓練して確かめる．

- **データ**：内側の円（クラス 0）と外側の輪（クラス 1）．入力の空間では直線で分けられない．
- **モデル**：隠れ層が 2 つ（8 ユニット → 2 ユニット）の MLP．2 つ目の隠れ層を 2 ユニットにするのは，その出力をそのまま平面に描くため．
  隠れ層の活性化関数は，出力が $-1$〜$1$ に収まって図が見やすい tanh を使う（ここだけ）．出力層はシグモイド．
- 訓練の中身（逆伝播）は第3回で学ぶ．ここでは **実行して図を見る** だけでよい．

In [ ]:
# データ：内側の円（クラス0）と外側の輪（クラス1）．各 200 点
n = 200
radius = np.concatenate([rng.uniform(0.0, 1.0, n), rng.uniform(1.6, 2.6, n)])   # 原点からの距離
angle = rng.uniform(0, 2 * np.pi, 2 * n)                                        # 角度
X_c = np.stack([radius * np.cos(angle), radius * np.sin(angle)], axis=1)        # 400×2（1 行が 1 点）
y_c = np.concatenate([np.zeros(n), np.ones(n)])                                 # 正解（0 か 1）
print("X_c の形 =", X_c.shape, " y_c の形 =", y_c.shape)

In [ ]:
# MLP の訓練と描画：【チャレンジ】コードを読み解いてみよう
def train_mlp(X, y, act='tanh', steps=3000, lr=0.5, seed=0):
    # act='tanh'：隠れ層に tanh を使う．act='identity'：活性化関数なし（何もしない）
    r = np.random.default_rng(seed)
    h = np.tanh if act == 'tanh' else (lambda u: u)
    dh = (lambda z: 1 - z ** 2) if act == 'tanh' else (lambda z: np.ones_like(z))   # h の微分
    W1 = r.normal(size=(8, 2)) / np.sqrt(2); b1 = np.zeros(8)     # 入力 2 → 隠れ 8
    W2 = r.normal(size=(2, 8)) / np.sqrt(8); b2 = np.zeros(2)     # 隠れ 8 → 隠れ 2
    w3 = r.normal(size=2) / np.sqrt(2); b3 = 0.0                  # 隠れ 2 → 出力 1（線形識別器）
    N = len(y)
    losses = []
    for t in range(steps):
        # 順伝播．X は 1 行が 1 データなので，X @ W1.T は「各行に W1 を掛ける」のと同じ
        z1 = h(X @ W1.T + b1)
        z2 = h(z1 @ W2.T + b2)
        p = sigmoid(z2 @ w3 + b3)
        # 損失：交差エントロピー
        losses.append(-np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)))
        # 逆伝播（第3回）．出力層の誤差信号 p − y から始めて入力側へ
        d3 = (p - y) / N
        d2 = np.outer(d3, w3) * dh(z2)
        d1 = (d2 @ W2) * dh(z1)
        w3 -= lr * (z2.T @ d3); b3 -= lr * d3.sum()
        W2 -= lr * (d2.T @ z1); b2 -= lr * d2.sum(axis=0)
        W1 -= lr * (d1.T @ X); b1 -= lr * d1.sum(axis=0)
    z2 = h(h(X @ W1.T + b1) @ W2.T + b2)
    p = sigmoid(z2 @ w3 + b3)
    return dict(z2=z2, p=p, w3=w3, b3=b3, losses=np.array(losses))

def show(X, y, res, title=''):
    acc = np.mean((res['p'] > 0.5) == (y == 1))
    print(f"{title} 最終損失 {res['losses'][-1]:.3f}, 正解率 {acc:.3f}")
    z2, w3, b3 = res['z2'], res['w3'], res['b3']
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    ax[0].scatter(X[y == 0, 0], X[y == 0, 1], s=12, label='class 0')
    ax[0].scatter(X[y == 1, 0], X[y == 1, 1], s=12, marker='^', label='class 1')
    ax[0].set_title('(a) input space x'); ax[0].set_aspect('equal'); ax[0].legend()
    ax[1].scatter(z2[y == 0, 0], z2[y == 0, 1], s=12)
    ax[1].scatter(z2[y == 1, 0], z2[y == 1, 1], s=12, marker='^')
    lo, hi = z2.min() - 0.2, z2.max() + 0.2
    xs = np.linspace(lo, hi, 2)
    if abs(w3[1]) > 1e-6:                                   # 出力層の境界 w3・z + b3 = 0 を破線で描く
        ax[1].plot(xs, -(w3[0] * xs + b3) / w3[1], 'k--', label='output layer: w.z + b = 0')
    ax[1].set_xlim(lo, hi); ax[1].set_ylim(lo, hi); ax[1].set_title('(b) hidden layer output z'); ax[1].legend()
    ax[2].plot(res['losses']); ax[2].set_title('(c) training loss'); ax[2].set_xlabel('step')
    plt.tight_layout(); plt.show()

In [ ]:
res = train_mlp(X_c, y_c, act='tanh')
show(X_c, y_c, res, 'tanh:')

**図の見方**
- (a) 入力の空間：● と ▲ が輪になっていて，直線では分けられない．
- (b) 2 つ目の隠れ層の出力：● と ▲ が離れ，破線 1 本で分かれている．破線は，出力層が引いている直線 $\boldsymbol{w}^\top\boldsymbol{z}+b=0$．
- (c) 損失：下がっていれば学習が進んでいる．

隠れ層が「直線で分けられる形」に変換し，出力層は直線を 1 本引いただけ，ということが分かる．

**やってみよう 3-2**　隠れ層の活性化関数をなくす（`act='identity'`）と何が起きるか．先に予想してから実行する．

In [ ]:
res_lin = train_mlp(X_c, y_c, act='tanh')      # TODO: 'tanh' を 'identity' に変えて実行する
show(X_c, y_c, res_lin, 'やってみよう 3-2:')

In [ ]:
# 解答
res_lin = train_mlp(X_c, y_c, act='identity')
show(X_c, y_c, res_lin, 'identity:')
# (b) でも ● と ▲ が混ざったままで，直線では分けられない．正解率は 0.5 前後（でたらめに答えた場合と同じ）にとどまる．
# 活性化関数がなければ，何層重ねても 1 層と同じ計算しかできず（p.7），境界は入力の空間でも直線にしかならないため．

## §4 損失，ソフトマックスと交差エントロピー（p.14–16）

損失は「正解とのずれ」を 1 つの数で表したもの（p.14）．

- 数を当てる（回帰）：二乗誤差 $L=\frac{1}{2}(\hat y-y)^2$
- 種類を当てる（分類）：交差エントロピー $L=-\log p_y$（$p_y$：正解のクラスの確率．$\log$ は自然対数で，NumPy では `np.log`）

In [ ]:
# 二乗誤差（p.14 の例）
y_hat, y_true = 2.5, 4.0
print("二乗誤差 L =", 0.5 * (y_hat - y_true) ** 2)          # 1.125

# 交差エントロピー（p.13 の例）：正解のクラスの確率 p_y が高いほど損失は小さい
for p_y in (0.2, 0.8, 0.99, 1.0):
    print(f"p_y = {p_y}: L = {-np.log(p_y) + 0.0:.3f}")     # 1.609, 0.223, 0.010, 0.000

### ソフトマックス（p.16）

分類では，出力層の総入力 $\boldsymbol{u}$（**ロジット**）を，足して 1 になる確率 $\boldsymbol{p}$ に変換する．

$$p_k=\frac{\exp(u_k)}{\sum_j \exp(u_j)},\qquad L=-\log p_y$$

In [ ]:
def softmax(u):
    u = u - u.max()          # 大きな値で計算があふれないようにする（結果は変わらない．やってみよう 4-1）
    e = np.exp(u)
    return e / e.sum()

def cross_entropy(u, y):     # y は正解クラスの番号（0 から数える）
    return -np.log(softmax(u)[y])

u = np.array([0.0, 2.0])
print("exp(u) =", np.exp(u), " 合計 =", np.exp(u).sum())     # (1, 7.39)，合計 8.39（p.15 の例）
print("p =", softmax(u))                                     # (0.119, 0.881)
print("正解が 2 番目（番号 1）のとき L =", cross_entropy(u, 1))   # -log 0.881 = 0.127
print("正解が 1 番目（番号 0）のとき L =", cross_entropy(u, 0))   # -log 0.119 = 2.127

**やってみよう 4-1**　ロジット $\boldsymbol{u}$ の全成分に同じ定数を足しても $\boldsymbol{p}$ は変わらない（ソフトマックスは成分の差だけを見る）．先に予想してから，定数を変えて確かめる．

In [ ]:
for c in (0.0, 5.0, -100.0):   # TODO: 別の値を足す
    print(f"u + {c:6.1f}: p = {softmax(u + c)}")

## §5 誤差信号 $\boldsymbol{\delta}=\boldsymbol{p}-\boldsymbol{y}$ の検証（p.17）

誤差信号は「出力層の総入力 $\boldsymbol{u}$ を少し変えたとき，損失がどれだけ変わるか」（損失の $\boldsymbol{u}$ についての微分）．
ソフトマックス＋交差エントロピーでは，これが $\boldsymbol{p}-\boldsymbol{y}$ になる（$\boldsymbol{y}$ は正解クラスだけ 1 の 1-hot ベクトル）．

本当にそうなるかを **数値微分** で確かめる：$\boldsymbol{u}$ の $k$ 番目だけを少し増やしたときと減らしたときの損失の差を，動かした幅で割る（＝傾き）．

In [ ]:
def numerical_grad(f, u, eps=1e-5):
    # f：u を入れると損失を返す関数．u の各成分について傾きを測る
    g = np.zeros_like(u)                             # u と同じ形の，0 で埋めた配列
    for k in range(len(u)):
        e = np.zeros_like(u); e[k] = eps             # k 番目だけ eps のベクトル
        g[k] = (f(u + e) - f(u - e)) / (2 * eps)     # 少し増やしたときと減らしたときの差 ÷ 幅
    return g

u5 = np.array([0.5, -1.0, 2.0, 0.0])     # 4 クラスのロジット
y5 = 2                                   # 正解クラスの番号（0 から数えるので 3 番目）
y_onehot = np.eye(4)[y5]                 # 正解の位置だけ 1 のベクトル (0, 0, 1, 0)
p5 = softmax(u5)

def loss_of_u(v):                        # 「u を入れると損失を返す関数」
    return cross_entropy(v, y5)

g_num = numerical_grad(loss_of_u, u5)
print("p - y        =", p5 - y_onehot)
print("数値微分     =", g_num)
print("最大誤差     =", np.abs(p5 - y_onehot - g_num).max(), "（1e-8 程度以下なら一致とみなす）")

**やってみよう 5-1（実行して結果を読む）**　二値分類（シグモイド＋二値交差エントロピー）でも，誤差信号が $p - y$ になることを同じ方法で確かめる．
二値交差エントロピー：正解が 1 なら $-\log p$，0 なら $-\log(1-p)$．まとめて $L=-y\log p-(1-y)\log(1-p)$．

**やってみよう 5-2（実行して結果を読む）**　p.15 の表の「違う行」を組み合わせた例．シグモイドに**二乗誤差** $L=\frac{1}{2}(p-y)^2$ を組み合わせる．
正解が $y=0$ なのに $u=10$（$p\approx1$．大きく外している）のとき，誤差信号がほぼ 0 になってしまうことを見る．

In [ ]:
def bce(u, y):                           # 二値交差エントロピー（u はスカラー）
    p = sigmoid(u)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

# 5-1：シグモイド＋二値交差エントロピー
u0, y0 = np.array([1.5]), 1.0            # numerical_grad に渡すため，u は長さ 1 の配列にしている
print("5-1  p - y =", sigmoid(u0) - y0, " 数値微分 =", numerical_grad(lambda v: bce(v[0], y0), u0))

# 5-2：シグモイド＋二乗誤差（違う行の組）
def sq_err(u, y):
    return 0.5 * (sigmoid(u) - y) ** 2

for uu in (0.0, 10.0):
    g_bad = numerical_grad(lambda v: sq_err(v[0], 0.0), np.array([uu]))
    g_good = sigmoid(uu) - 0.0           # 同じ行の組（シグモイド＋二値交差エントロピー）なら p − y
    print(f"5-2  u={uu:4.1f}, 正解 y=0, p={sigmoid(uu):.4f}: 二乗誤差の誤差信号 = {g_bad[0]:.2e}，同じ行の組なら {g_good:.2f}")
# 解答：u=10 では p≈1 で大きく外しているのに，二乗誤差との組では誤差信号が 4.5e-05 とほぼ 0．
#       これでは重みがほとんど直らない．同じ行の組なら 1.00 で，しっかり直る．これが「同じ行から選ぶ」理由．

## §6 埋め込み（p.18）

$V$ 種類の記号に，長さ $d$ のベクトルを割り当てる表 $E$（$d\times V$．第 $i$ 列が記号 $i$ のベクトル $\boldsymbol{e}_i$）．
記号 $i$ の 1-hot ベクトルに $E$ を掛ける（$E\times$ 1-hot）と，第 $i$ 列が取り出される．

ここでは学習済みのつもりの小さな表を手で作る．近い記号を探すのには **コサイン類似度**（2 つのベクトルの向きの近さ．1 に近いほど似ている）を使う．

In [ ]:
vocab = ['りんご', 'みかん', 'ぶどう', 'ノートPC', 'スマホ', 'キーボード']
V, d = len(vocab), 4
# 書きやすいよう記号ごとに 1 行で書き，最後に .T で転置して「列＝記号」にする
E = np.array([[0.9, 0.1, 0.0, 0.2],     # りんご
              [0.8, 0.2, 0.1, 0.1],     # みかん
              [0.7, 0.3, 0.0, 0.3],     # ぶどう
              [0.0, 0.1, 0.9, 0.8],     # ノートPC
              [0.1, 0.0, 0.8, 0.9],     # スマホ
              [0.0, 0.2, 0.9, 0.6]]).T  # キーボード
print("E の形 =", E.shape, "（d×V），パラメータ数 =", E.size)

i = vocab.index('ノートPC')              # ノートPC の番号
onehot = np.eye(V)[i]                    # i 番目だけ 1 のベクトル
print("E × 1-hot =", E @ onehot, " ＝ E の第 i 列 =", E[:, i])

def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))   # 内積 ÷（長さ × 長さ）

def nearest(E, vocab, word, k=2):
    # word に近い記号を，コサイン類似度の高い順に k 個返す（自分自身は除く）
    i = vocab.index(word)
    sims = np.array([cosine(E[:, i], E[:, j]) for j in range(len(vocab))])
    order = np.argsort(-sims)            # 類似度の高い順に並べた番号
    return [(vocab[j], round(float(sims[j]), 2)) for j in order if j != i][:k]

for word in vocab:
    print(f"{word} に近い: {nearest(E, vocab, word)}")

### 埋め込みは MLP の入口の層（p.18）

記号は，埋め込みでベクトルに変えてから次の層（MLP）に渡す．1-hot ベクトルに $E$ を掛けるのと，表の列を取り出すのは同じ計算なので，
埋め込みは「重み行列が $E$ の層（バイアスと活性化関数なし）」である．$E$ も，後段の重みと一緒に学習する．

In [ ]:
W_next = rng.normal(size=(3, d))            # 埋め込みの次の層（3 ユニット）の重み行列．ここでは乱数
b_next = np.zeros(3)

for word in ['りんご', 'みかん', 'ノートPC']:
    i = vocab.index(word)
    e = E[:, i]                             # 埋め込み：表の第 i 列を取り出す（＝ E @ 1-hot）
    z_next = relu(W_next @ e + b_next)      # 次の層：これまでと同じ h(W z + b)
    print(f"{word}: 埋め込み {e} → 次の層の出力 {z_next}")
# 似た埋め込み（りんご・みかん）は，次の層の出力も似る

**やってみよう 6-1**　新しい記号「バナナ」を追加する．果物の仲間に近くなるような 4 つの数を自分で決めて，最も近い記号が果物になることを確かめる．

In [ ]:
banana = np.array([0.0, 0.0, 0.0, 1.0])     # TODO: 果物（りんご・みかん・ぶどう）に近くなるよう 4 つの数を書き換える
E_new = np.concatenate([E, banana[:, None]], axis=1)   # E の右端に 1 列足す
vocab_new = vocab + ['バナナ']
print("バナナ に近い:", nearest(E_new, vocab_new, 'バナナ'))

In [ ]:
# 解答例：果物の列は 1 番目の成分が大きい（0.7〜0.9）ので，それに合わせる
banana = np.array([0.8, 0.2, 0.1, 0.2])
E_new = np.concatenate([E, banana[:, None]], axis=1)
print("バナナ に近い:", nearest(E_new, vocab + ['バナナ'], 'バナナ'))

**やってみよう 6-2（実行して結果を読む）**　記号が 50,000 種類，ベクトルの長さが 768 のとき，埋め込み行列のパラメータ数はいくつか（p.20 の問 4：大きさは $d\times V$）．

In [ ]:
print("768 × 50000 =", 768 * 50000)   # 3,840 万個．文章を扱うモデルでは，入口の表だけでこの規模になる

## §7（任意）PyTorch で同じことをする（テキスト付録B）

PyTorch が入っていれば，§3 の MLP・§4 の損失・§6 の埋め込みが `nn.Linear`，`nn.CrossEntropyLoss`，`nn.Embedding` にそのまま対応することを見る．
実装上の注意：`nn.CrossEntropyLoss` はロジットを直接受け取る（中でソフトマックスを計算する）．確率にしてから渡すと値がずれる．

In [ ]:
try:
    import torch
    import torch.nn as nn
    torch.manual_seed(0)
    mlp = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))
    print("パラメータ数 =", sum(p.numel() for p in mlp.parameters()))   # 203530（§3 と一致）

    logits = torch.tensor([[0.0, 2.0]])
    target = torch.tensor([1])
    print("CE(logits) =", nn.CrossEntropyLoss()(logits, target).item())                    # 0.127（§4 と一致）

    emb = nn.Embedding(num_embeddings=6, embedding_dim=4)
    print("Embedding の重みの形 =", tuple(emb.weight.shape), "（PyTorch は V×d で持つ）")
    print("記号 3 のベクトル =", emb(torch.tensor([3])).detach().numpy())
except ImportError:
    print("PyTorch が見つからないので，この節は読み飛ばす．")